# Pandas `groupby()`

## 1. What is `groupby()`?

In pandas, `groupby()` is used to **split a DataFrame into groups based on one or more columns**. After creating groups, we can inspect each group or calculate summary values such as the minimum, maximum, mean, count, and sum.

For example, if a DataFrame contains students' marks, we can group the rows by `Name` to view each student's records or calculate each student's average marks.

Pandas `groupby()` follows the **split–apply–combine** process:

1. **Split:** Divide the rows into groups based on a column's values.
2. **Apply:** Perform an operation on each group, such as `mean()` or `max()`.
3. **Combine:** Return the results together in a Series or DataFrame.

> **Note:** `groupby()` creates a grouped object. It does not automatically calculate a summary until you apply an operation or inspect the groups.

## 2. Create a sample DataFrame

We will use a small DataFrame containing student names and their marks in Maths and English. Some names appear more than once, so we can see how rows are grouped.

In [3]:
import pandas as pd

df = pd.DataFrame({
    "Name": ["a", "b", "c", "a", "b", "c", "a", "b", "d", "d"],
    "Maths": [44, 23, 67, 22, 87, 23, 77, 34, 78, 52],
    "English": [74, 93, 67, 22, 34, 23, 77, 34, 78, 52]
})

df

,Name,Maths,English
0,a,44,74
1,b,23,93
2,c,67,67
3,a,22,22
4,b,87,34
5,c,23,23
6,a,77,77
7,b,34,34
8,d,78,78
9,d,52,52


## 3. Group rows using one column

### Syntax

```python
df.groupby("column_name")
```

### Example

Group the rows by `Name`. Every unique name becomes a group, and all rows with that name belong to the same group.

In [4]:
grouped_data = df.groupby("Name")
grouped_data

The result is a `DataFrameGroupBy` object. To see the actual rows in each group, iterate over it or use `get_group()`.

## 4. Display every group using a loop

When iterating over a grouped object, pandas provides two values:
- `name`: the group key (for example, `"a"`).
- `data`: the DataFrame containing the rows for that group.

This is useful when you want to inspect each group separately.

In [5]:
for name, data in grouped_data:
    print(f"\nGroup: {name}")
    print(data)


Group: a
  Name  Maths  English
0    a     44       74
3    a     22       22
6    a     77       77

Group: b
  Name  Maths  English
1    b     23       93
4    b     87       34
7    b     34       34

Group: c
  Name  Maths  English
2    c     67       67
5    c     23       23

Group: d
  Name  Maths  English
8    d     78       78
9    d     52       52


## 5. Get one specific group with `get_group()`

Use `get_group(value)` to retrieve the rows for one group. The value must match a group key.

For example, the following returns only the rows where `Name` is `"a"`.

In [6]:
grouped_data.get_group("a")

,Name,Maths,English
0,a,44,74
3,a,22,22
6,a,77,77


## 6. Find the minimum value with `min()`

`min()` returns the minimum value in each group for each applicable column. In this example, pandas calculates the minimum `Maths` and `English` marks separately for each name.

In [7]:
grouped_data.min()

,Maths,English
Name,,
a,22,22
b,23,34
c,23,23
d,52,52


## 7. Find the maximum value with `max()`

`max()` returns the maximum value in each group for each applicable column.

In [8]:
grouped_data.max()

,Maths,English
Name,,
a,77,77
b,87,93
c,67,67
d,78,78


## 8. Calculate the average with `mean()`

`mean()` calculates the average for each numeric column within each group. For example, the `Maths` result for name `"a"` is the average of 44, 22, and 77.

The group key (`Name`) becomes the index of the result.

In [9]:
grouped_data.mean()

,Maths,English
Name,,
a,47.666667,57.666667
b,48.000000,53.666667
c,45.000000,45.000000
d,65.000000,65.000000


## 9. Other useful aggregation methods

You can apply different aggregation methods to a grouped object:

- `sum()` — adds values in each group.
- `count()` — counts non-missing values in each column.
- `size()` — counts rows in each group, including rows with missing values.
- `median()` — calculates the median.
- `std()` — calculates the standard deviation.
- `agg()` — applies one or more aggregation functions.

These methods are useful for summarizing datasets quickly.

In [10]:
# Total marks in each group
grouped_data.sum()

,Maths,English
Name,,
a,143,173
b,144,161
c,90,90
d,130,130


In [11]:
# Number of non-missing values in each column
grouped_data.count()

,Maths,English
Name,,
a,3,3
b,3,3
c,2,2
d,2,2


In [12]:
# Number of rows in each group
grouped_data.size()

Name
a    3
b    3
c    2
d    2
dtype: int64

In [13]:
# Apply multiple summary functions to numeric columns
grouped_data[["Maths", "English"]].agg(["min", "max", "mean"])

Maths                English               
       min max       mean     min max       mean
Name                                            
a       22  77  47.666667      22  77  57.666667
b       23  87  48.000000      34  93  53.666667
c       23  67  45.000000      23  67  45.000000
d       52  78  65.000000      52  78  65.000000

## 10. Convert groups into a list

Calling `list()` on a grouped object creates a list of `(group_name, group_dataframe)` pairs. This can help when you need to access the groups as ordinary Python objects.

Each item in the list contains:
1. The group key.
2. The DataFrame for that group.

For most analysis tasks, it is usually simpler to work directly with the grouped object rather than converting it to a list.

In [14]:
groups_list = list(grouped_data)

# Display the first pair: group key and its DataFrame
groups_list[0]

('a',
   Name  Maths  English
 0    a     44       74
 3    a     22       22
 6    a     77       77)

## 11. Group by more than one column

You can pass a list of column names to `groupby()`. Pandas then creates groups for each unique combination of those column values.

General syntax:

```python
df.groupby(["column_1", "column_2"])
```

This sample dataset has no second category column, so the example below adds a `Class` column to demonstrate the syntax.

In [15]:
df_with_class = df.copy()
df_with_class["Class"] = [
    "A", "A", "B", "B", "A", "B", "A", "B", "A", "B"
]

df_with_class.groupby(["Class", "Name"])[["Maths", "English"]].mean()

Maths  English
Class Name                
A     a      60.5     75.5
      b      55.0     63.5
      d      78.0     78.0
B     a      22.0     22.0
      b      34.0     34.0
      c      45.0     45.0
      d      52.0     52.0

## 12. Select a column before aggregating

If you need a summary for only one numeric column, select it before applying the aggregation. This keeps the output focused.

For example, to calculate the average Maths marks for each name:

In [16]:
grouped_data["Maths"].mean()

Name
a    47.666667
b    48.000000
c    45.000000
d    65.000000
Name: Maths, dtype: float64

## 13. Key takeaways

- `df.groupby("column")` groups rows based on unique values in a column.
- `groupby()` alone creates a grouped object; apply a method to calculate a summary.
- Use a loop to inspect all groups.
- Use `get_group()` to retrieve one group.
- Use `min()`, `max()`, `mean()`, `sum()`, `count()`, and `size()` to summarize groups.
- Use `agg()` to calculate multiple summaries.
- Pass a list of columns to group by multiple columns.

### Quick practice

1. Find the maximum English marks for each name.
2. Calculate the total Maths marks for each name.
3. Display only the rows for group `"c"`.
4. Calculate the average marks grouped by `Class` and `Name`.

Try solving these before checking the examples above again.